In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

In [3]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix, hstack

# 일반 Feature
# X_full = pd.read_pickle("../data/processed/X_full.pkl")
X_final = pd.read_pickle("../data/processed/X_final.pkl")

# Target
y = pd.read_pickle("../data/processed/y.pkl")

# print("X_full :", X_full.shape)
print("X_final:", X_final.shape)
print("y_     :", y.shape)

X_final: (1481611, 297)
y_     : (1481611,)


In [4]:
y_log = np.log1p(y)
print("y_log      :", y_log.shape)

y_log      : (1481611,)


In [5]:
# y로그변환 저장
y_log.to_pickle("../data/processed/y_log.pkl")

In [1]:
import pandas as pd
import numpy as np

df = pd.read_pickle(
    "../data/processed/mercari_preprocessed.pkl"
)

df.shape

(1481611, 14)

In [2]:
print("원본 df:", df.shape[0])
# print("X_full:", X_full.shape[0])
# print("차이:", df.shape[0] - X_full.shape[0])

# price == 0 인 행이 몇 개인지 확인
print("price == 0 개수:", (df['price'] == 0).sum())

원본 df: 1481611
price == 0 개수: 0


In [3]:
tfidf_all = pd.read_pickle(
    "../data/processed/tfidf_all.pkl"
)

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

# 현재 df(1481611행) 기준으로 다시 벡터화
tfidf_vectorizer = TfidfVectorizer(max_features=50000)
tfidf_all = tfidf_vectorizer.fit_transform(df["text"])

print(tfidf_all.shape)  # (1481611, 50000) 나와야 정상

(1481611, 50000)


In [6]:
from scipy.sparse import save_npz
save_npz("../data/processed/tfidf_all_1481611.npz", tfidf_all)

In [ ]:
# # 순서가 같은 df에서 나온 것인지 최종 확인
# print(df.index[:5])
# print(X_full.index[:5])  # X_full이 DataFrame이라면

Index([0, 1, 2, 3, 4], dtype='int64')
Index([0, 1, 2, 3, 4], dtype='int64')


In [7]:
import numpy as np
from scipy.sparse import hstack, csr_matrix, save_npz

X_combined_final = hstack([
    tfidf_all,
    csr_matrix(X_final.values.astype(np.float32))
]).tocsr()

save_npz("../data/processed/X_combined_final.npz", X_combined_final)
print(X_combined_final.shape)

(1481611, 50297)


메모리 때문에 리로드 후 아래 실행

In [1]:
#릿지 
from scipy.sparse import load_npz
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
import numpy as np

X_combined_final = load_npz("../data/processed/X_combined_final.npz")
y_log = pd.read_pickle("../data/processed/y_log.pkl")

X_train, X_test, y_train_log, y_test_log = train_test_split(
    X_combined_final, y_log, test_size=0.2, random_state=42
)

model = Ridge(alpha=1.0)
model.fit(X_train, y_train_log)

pred_log = model.predict(X_test)
pred = np.expm1(pred_log)        # 예측값 원래 스케일로 복원
pred = np.clip(pred, 0, None)    # 음수 방지

y_test_orig = np.expm1(y_test_log)  # 정답도 원래 스케일로 복원

def rmsle(y_true, y_pred):
    y_pred = np.clip(y_pred, 0, None)
    return np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_true))**2))

print("RMSLE (log1p 학습):", rmsle(y_test_orig, pred))

RMSLE (log1p 학습): 0.4962310734486018


In [2]:
import joblib

# 모델 저장 경로 설정 (폴더가 존재한다고 가정)
model_path = "../data/processed/base_ridge_model.pkl"

# joblib을 이용해 모델 객체 저장
joblib.dump(model, model_path)

['../data/processed/base_ridge_model.pkl']